In [1]:
import numpy as np
import pandas as pd
import torch

In [2]:
path = "/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/amber/combined_stage_label_with_evidence_and_mlp_09_11_2025.pkl"

In [3]:


res_data = pd.read_pickle(path)

In [4]:
res_data.head(4)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,candidates,hallucination_candidates,labels_with_evidence
0,Describe this image.,The image depicts a group of four people walki...,1,AMBER_1.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,None,no_datatype,[],[],"[{'word': 'group', 'evidence': [0.0012769105, ..."
1,Describe this image.,The image features a man wearing a life jacket...,2,AMBER_2.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,None,no_datatype,[],[],"[{'word': 'man', 'evidence': [0.008708082, 0.0..."
2,Describe this image.,The image features a young child standing in a...,3,AMBER_3.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,None,no_datatype,[],[],"[{'word': 'young', 'evidence': [0.17931546, 0...."
3,Describe this image.,The image features a woman wearing a white shi...,4,AMBER_4.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,None,no_datatype,[],[],"[{'word': 'woman', 'evidence': [0.13150519, 0...."


In [47]:

all_seqs = []
ids = []
for inx, row in res_data.iterrows():
    res = row["labels_with_evidence"]
    seq = row["answer"]
    hal_words = []
    for i in res:
        viz_evi = (torch.tensor(i["evidence"]) >= 0.5).int().sum().item()
        prob = i["label"]
        if prob <= 0.5:
            hal_words.append(i["word"])
        # if viz_evi <= 1 and prob <= 0.92:
        #     hal_words.append(i["word"])
    for hw in hal_words:
            seq = seq.replace(hw.strip().strip("."), "")
    all_seqs.append(seq)
    ids.append(row["question_id"])
    
req_df = pd.DataFrame({"id": ids, "response": all_seqs})

req_df.to_json("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/amber/amber_post_halu_res.json", lines=True, orient="records")